In [6]:
import numpy as np
import string
import glob
import os
import tensorflow as tf
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add, RepeatVector
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tqdm import tqdm
os.environ['TQDM_DISABLE'] = '1'

In [4]:
def load_doc(filename):
  with open(filename, 'r') as f:
    text = f.read()
  return text

filename = 'Flickr8k.token.txt'
doc = load_doc(filename)

desc = {}
for line in doc.strip().split('\n'):
  tokens = line.split('\t')
  img_id, img_desc = tokens[0], tokens[1]
  img_id = img_id.split('#')[0]
  img_desc = img_desc.lower().translate(str.maketrans('', '', string.punctuation))
  if img_id not in desc:
    desc[img_id] = []
  desc[img_id].append('startseq' + img_desc + 'endseq')

print(f"Loaded {len(desc)} image descriptions.")

Loaded 8092 image descriptions.


In [5]:
model = InceptionV3(weights='imagenet')
model = Model(model.input, model.layers[-2].output)

features = {}
images = glob.glob('Flicker8k_Dataset/*.jpg')

for img_path in tqdm(images, disable=False):
  img_id = os.path.basename(img_path).split('.')[0]
  img = image.load_img(img_path, target_size=(299, 299))
  img = image.img_to_array(img)
  img = np.expand_dims(img, axis=0)
  img = preprocess_input(img)
  feature = model.predict(img, verbose=0)
  features[img_id] = feature

print(f"Extracted features for {len(features)} images.")

I0000 00:00:1751982546.219763    2538 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1767 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
  0%|          | 0/8091 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1751982550.590707    7494 service.cc:152] XLA service 0x7f62380017c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1751982550.590758    7494 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2025-07-08 19:19:10.710337: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1751982551.681404    7494 cuda_dnn.cc:529] Loaded cuDNN version 90800
2025-07-08 19:19:14.284226: W external/local_xla/

Extracted features for 8091 images.


In [7]:
descs = []
for key in desc:
  [descs.append(d) for d in desc[key]]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(descs)
vocab_size = len(tokenizer.word_index) + 1
maxLen = max(len(d.split()) for d in descs)
print(f"Vocabulary Size: {vocab_size}")
print(f"Max Caption Length: {maxLen}")

Vocabulary Size: 10279
Max Caption Length: 37


In [29]:
def create_tf_dataset(desc, features, tokenizer, maxLen, vocab_size, batch_size):
    def data_generator():
        while True:
            for key, value in desc.items():
                if key in features:
                    feature = features[key][0]
                    for d in value:
                        seq = tokenizer.texts_to_sequences([d])[0]
                        for i in range(1, len(seq)):
                            in_seq, out_seq = seq[:i], seq[i]
                            in_seq = pad_sequences([in_seq], maxlen=maxLen)[0]
                            out_seq_onehot = to_categorical(out_seq, num_classes=vocab_size)
                            yield (feature, in_seq), out_seq_onehot
    feature_dim = features[list(features.keys())[0]][0].shape[0]
    output_signature = (
        (
            tf.TensorSpec(shape=(feature_dim,), dtype=tf.float32),
            tf.TensorSpec(shape=(maxLen,), dtype=tf.int32)
        ),
        tf.TensorSpec(shape=(vocab_size,), dtype=tf.float32)
    )
    dataset = tf.data.Dataset.from_generator(data_generator, output_signature=output_signature)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [46]:
inputs1 = Input(shape=(2048,))
feature1 = Dropout(0.5)(inputs1)
feature2 = Dense(256, activation='relu')(feature1)

inputs2 = Input(shape=(maxLen,))
emb1 = Embedding(vocab_size, 256)(inputs2)
emb2 = Dropout(0.5)(emb1)
feature_seq = RepeatVector(maxLen)(feature2)

combined = add([feature_seq, emb2])

LSTM_out = LSTM(256)(combined)
decoder1 = Dense(256, activation='relu')(LSTM_out)
outputs = Dense(vocab_size, activation='softmax')(decoder1)

model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 2048)      │          0 │ input_layer_7[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_8       │ (None, 34)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 256)       │    524,544 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 34, 256)   │  2,631,424 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_3     │ (None, 34, 256)   │          0 │ dense_9[0][0]     │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 34, 256)   │          0 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 34, 256)   │          0 │ repeat_vector_3[… │
│                     │                   │            │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 256)       │    525,312 │ add_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 256)       │     65,792 │ lstm_3[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 10279)     │  2,641,703 │ dense_10[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,388,775 (24.37 MB)

 Trainable params: 6,388,775 (24.37 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
batch_size = 32
epochs = 20

def normalize_desc_keys(desc):
    normalized_desc = {}
    for key, value in desc.items():
        new_key = key.replace('.jpg', '')
        normalized_desc[new_key] = value
    return normalized_desc

desc_normalized = normalize_desc_keys(desc)

common_keys = set(desc_normalized.keys()).intersection(set(features.keys()))
filtered_desc = {key: desc_normalized[key] for key in common_keys}

def calculate_steps(desc, batch_size):
    total_seq = 0
    for key, value in desc.items():
        for d in value:
            seq_len = len(tokenizer.texts_to_sequences([d])[0])
            total_seq += max(1, seq_len - 1)
    return total_seq // batch_size

steps = calculate_steps(filtered_desc, batch_size)
print(f"Total Steps per Epoch: {steps}")
print(f"Number of images: {len(filtered_desc)}")
print(f"Total sequences: {steps * batch_size}")

callbacks = [
    ModelCheckpoint('best_model.h5', save_best_only=True, monitor='loss'),
    EarlyStopping(patience=5, monitor='loss'),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7)
]

dataset = create_tf_dataset(filtered_desc, features, tokenizer, maxLen, vocab_size, batch_size)

history = model.fit(
    dataset,
    epochs=epochs,
    steps_per_epoch=steps,
    callbacks=callbacks,
    verbose=1
)

Total Steps per Epoch: 13520
Number of images: 8091
Total sequences: 432640
Epoch 1/20
13520/13520 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3200 - loss: 3.5574

13520/13520 ━━━━━━━━━━━━━━━━━━━━ 350s 26ms/step - accuracy: 0.3200 - loss: 3.5574 - learning_rate: 0.0010
Epoch 2/20
13518/13520 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3279 - loss: 3.4762

13520/13520 ━━━━━━━━━━━━━━━━━━━━ 336s 25ms/step - accuracy: 0.3279 - loss: 3.4762 - learning_rate: 0.0010
Epoch 3/20
13520/13520 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3335 - loss: 3.4168

13520/13520 ━━━━━━━━━━━━━━━━━━━━ 347s 26ms/step - accuracy: 0.3335 - loss: 3.4168 - learning_rate: 0.0010
Epoch 4/20
13519/13520 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3371 - loss: 3.3742

13520/13520 ━━━━━━━━━━━━━━━━━━━━ 349s 26ms/step - accuracy: 0.3371 - loss: 3.3742 - learning_rate: 0.0010
Epoch 5/20
13518/13520 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3401 - loss: 3.3454

...

13520/13520 ━━━━━━━━━━━━━━━━━━━━ 548s 41ms/step - accuracy: 0.3475 - loss: 3.2746 - learning_rate: 0.0010
Epoch 12/20
13519/13520 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3486 - loss: 3.2736

13520/13520 ━━━━━━━━━━━━━━━━━━━━ 495s 37ms/step - accuracy: 0.3486 - loss: 3.2736 - learning_rate: 0.0010
Epoch 13/20
13518/13520 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3488 - loss: 3.2696

13520/13520 ━━━━━━━━━━━━━━━━━━━━ 334s 25ms/step - accuracy: 0.3488 - loss: 3.2696 - learning_rate: 0.0010
Epoch 14/20
13519/13520 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3499 - loss: 3.2693

13520/13520 ━━━━━━━━━━━━━━━━━━━━ 331s 24ms/step - accuracy: 0.3499 - loss: 3.2693 - learning_rate: 0.0010
Epoch 15/20
13520/13520 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3494 - loss: 3.2644

13520/13520 ━━━━━━━━━━━━━━━━━━━━ 322s 24ms/step - accuracy: 0.3494 - loss: 3.2644 - learning_rate: 0.0010
Epoch 16/20
13520/13520 ━━━━━━━━━━━━━━━━━━━━ 369s 27ms/step - accuracy: 0.3487 - loss: 3.2693 - learning_rate: 0.0010
Epoch 17/20
13520/13520 ━━━━━━━━━━━━━━━━━━━━ 364s 27ms/step - accuracy: 0.3486 - loss: 3.2736 - learning_rate: 0.0010
Epoch 18/20
13520/13520 ━━━━━━━━━━━━━━━━━━━━ 497s 37ms/step - accuracy: 0.3493 - loss: 3.2720 - learning_rate: 0.0010
Epoch 19/20
13520/13520 ━━━━━━━━━━━━━━━━━━━━ 328s 24ms/step - accuracy: 0.3500 - loss: 3.2698 - learning_rate: 0.0010
Epoch 20/20
13520/13520 ━━━━━━━━━━━━━━━━━━━━ 343s 25ms/step - accuracy: 0.3488 - loss: 3.2726 - learning_rate: 0.0010


In [ ]:
def generate_desc(model, tokenizer, photo, max_length):
    in_text = 'startseq'
    for _ in range(max_length):
        sequence = tokenizer.texts_to_sequences([in_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_length)
        yhat = model.predict([photo, sequence], verbose=0)
        yhat = np.argmax(yhat)
        word = None
        for w, index in tokenizer.word_index.items():
            if index == yhat:
                word = w
                break
        if word is None:
            break
        in_text += ' ' + word
        if word == 'endseq':
            break
    return in_text

keys = list(features.keys())
photo = features[keys[1]]
print(generate_desc(model, tokenizer, photo, maxLen))
